In [ ]:
!pip install bertopic

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import torch
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
import umap
import hdbscan
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import json
import numpy as np
import os

In [ ]:
import re
import time
from tqdm import tqdm
from nltk.corpus import stopwords
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
import glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter
import re
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk import word_tokenize
from string import punctuation
from nltk.corpus import stopwords
from nltk import word_tokenize
import nltk
from string import punctuation
punctuation += '$£&'
from nltk.corpus import stopwords
EN_STOPWORDS = set(stopwords.words('english'))

In [ ]:
import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Define the main project directory created after extracting the zip file
BASE_PROJECT_DIR = "./TAM_2526_Project/TAM 2526 Project"

# Dataset folder
DATA_DIR = os.path.join(BASE_PROJECT_DIR, "data")

# Output folders
BASE_OUTPUT_DIR = os.path.join(BASE_PROJECT_DIR, "outputs")
FIGURES_DIR = os.path.join(BASE_OUTPUT_DIR, "figures")
TABLES_DIR = os.path.join(BASE_OUTPUT_DIR, "tables")
MODELS_DIR = os.path.join(BASE_OUTPUT_DIR, "models")
EMBEDDINGS_DIR = os.path.join(BASE_OUTPUT_DIR, "embeddings")

# Create folders if they do not exist
for folder in [BASE_OUTPUT_DIR, FIGURES_DIR, TABLES_DIR, MODELS_DIR, EMBEDDINGS_DIR]:
    os.makedirs(folder, exist_ok=True)

# Helper functions
def get_figure_path(filename):
    return os.path.join(FIGURES_DIR, filename)

def get_table_path(filename):
    return os.path.join(TABLES_DIR, filename)

def get_model_path(filename):
    return os.path.join(MODELS_DIR, filename)

def get_embedding_path(filename):
    return os.path.join(EMBEDDINGS_DIR, filename)

## **DATASET**

The dataset consists of 35 transcripts of Donald Trump rally speeches delivered between 2019 and 2020.  
Each speech is stored as a `.txt` file, and the filename contains basic metadata about the event, including the location, month, day, and year of the rally.

The first step of the analysis is to load all text files into a structured `pandas` DataFrame.  
For each speech, the notebook stores the original filename and the full transcript. Then, additional metadata are extracted from the filename through regular expressions:

- `year`: the year of the rally, either 2019 or 2020;
- `month`: the abbreviated month of the rally;
- `location`: the place where the rally was held.

This metadata is necessary for the later stages of the project, especially for comparing topic distributions across time and geographical locations.  
After loading and metadata extraction, the speeches are sorted chronologically to preserve the temporal structure of the corpus.

In [ ]:
#importing the dataset

import zipfile
import os

zip_path = "TAM 2526 Project.zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("TAM_2526_Project")


os.listdir("TAM_2526_Project")


for root, dirs, files in os.walk("."):
    if "data" in dirs:
        print(os.path.join(root, "data"))

archive_path = DATA_DIR

In [ ]:
# Retrieve all .txt files recursively from the dataset folder.
#in case the archive path doessn't work, substitute it with the result of the previous cell

archive_path = "./TAM_2526_Project/TAM 2526 Project/data"
files = glob.glob(archive_path + '**/*.txt', recursive=True)


data = []
for file in files:
    filename = os.path.basename(file)
    with open(file, 'r', encoding='utf-8') as f:
        text = f.read()
    data.append({'filename': filename, 'text': text})


df = pd.DataFrame(data)

print(f"There are {len(df)} speeches in the dataset.")

# Extract metadata
# 1. Extract the year from the filename and store it as an integer.
df['year'] = df['filename'].str.extract(r'(2019|2020)').astype(int)

# 2. Extract the month abbreviation from the filename.
# A numerical month column is temporarily created to sort speeches chronologically.
month_pattern = r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)'
df['month'] = df['filename'].str.extract(month_pattern)

month_map = {'Jan':1, 'Feb':2, 'Mar':3, 'Apr':4, 'May':5, 'Jun':6,
             'Jul':7, 'Aug':8, 'Sep':9, 'Oct':10, 'Nov':11, 'Dec':12}

# Sort speeches by month, then remove the temporary numerical month column.
df['month_num'] = df['month'].map(month_map)
df = df.sort_values(by=['year', 'month_num']).drop(columns=['month_num']).reset_index(drop=True)


# 3. Extract the rally location by removing month, year, day numbers, and file extension from the filename.
df['location'] = df['filename'].str.replace(r'(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)', '', regex=True)
df['location'] = df['location'].str.replace(r'(2019|2020)', '', regex=True)

df['location'] = df['location'].str.replace(r'\d{1,2}', '', regex=True)
df['location'] = df['location'].str.replace('.txt', '', regex=False)
df['location'] = df['location'].str.replace('_', ' ').str.strip().str.title()

df

In [ ]:
print(df['filename'][0])
print(df['text'][0])
print(df['month'][0])
print(df['year'][0])
print(df['location'][0])

## **INFORMATION ABOUT THE DATASET**

### Dataset inspection

This subsection provides a preliminary overview of the dataset structure.  
It checks the number of rows and columns, the available metadata fields, and the distribution of speeches by year, month, and location.


In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.index

In [ ]:
df['month'].value_counts()

In [ ]:
df['year'].value_counts()

In [ ]:
df['location'].value_counts()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Create a figure and a GridSpec layout
fig = plt.figure(figsize=(15, 15))
gs = gridspec.GridSpec(2, 2, figure=fig, height_ratios=[1, 1.5]) # 2 rows, 2 columns, with second row taller

# Plot for 'month'
ax0 = fig.add_subplot(gs[0, 0])
df['month'].value_counts().plot(kind='bar', ax=ax0, color='lightgreen', edgecolor='purple', linewidth=1.2)
ax0.set_title('Speeches per month')
ax0.set_xlabel('Month')
ax0.set_ylabel('Number of speeches')
ax0.grid(True, which='both', linestyle='--', linewidth=0.5)
ax0.tick_params(axis='x', rotation=45)

# Plot for 'year'
ax1 = fig.add_subplot(gs[0, 1])
df['year'].value_counts().plot(kind='bar', ax=ax1, color='lightgreen', edgecolor='purple', linewidth=1.2)
ax1.set_title('Speeches per year')
ax1.set_xlabel('Year')
ax1.set_ylabel('Number of speeches')
ax1.grid(True, which='both', linestyle='--', linewidth=0.5)
ax1.tick_params(axis='x', rotation=45)

# Plot for 'location'
ax2 = fig.add_subplot(gs[1, :])
df['location'].value_counts().plot(kind='barh', ax=ax2, color='lightgreen', edgecolor='purple', linewidth=1.2)
ax2.set_title('Speeches per location')
ax2.set_xlabel('Number of speeches')
ax2.set_ylabel('Location')
ax2.grid(True, which='both', linestyle='--', linewidth=0.5)
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import matplotlib.gridspec as gridspec

# month order for plotting
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
df['month'] = pd.Categorical(df['month'], categories=month_order, ordered=True)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, 2)

# Speeches by Year and Month
ax0 = fig.add_subplot(gs[0, 0])
counts_year_month = df.groupby(['month', 'year']).size().unstack(fill_value=0)
counts_year_month.plot(kind='bar', ax=ax0, rot=45, cmap='summer')
ax0.set_title('Number of speeches per month and year')
ax0.set_xlabel('Month')
ax0.set_ylabel('Number of speeches')
ax0.legend(title='Year')
ax0.grid(axis='y', linestyle='--', alpha=0.7)

# Speeches by Year and Location
ax1 = fig.add_subplot(gs[0, 1])
counts_year_location = df.groupby(['location', 'year']).size().unstack(fill_value=0)
total_location_counts = df['location'].value_counts()
locations_to_plot = total_location_counts[total_location_counts > 1].index.tolist()

if locations_to_plot:
    counts_year_location_filtered = counts_year_location.loc[locations_to_plot]
    sns.heatmap(counts_year_location_filtered, annot=True, fmt='d', cmap='summer', ax=ax1, linewidths=.5, linecolor='black')
else:
    sns.heatmap(counts_year_location, annot=True, fmt='d', cmap='summer', ax=ax1, linewidths=.5, linecolor='black')

ax1.set_title('Number of speeches per year and location')
ax1.set_xlabel('Year')
ax1.set_ylabel('Location')
ax1.tick_params(axis='x', rotation=45)
ax1.tick_params(axis='y', rotation=0)

# Speeches by Month and Location
ax2 = fig.add_subplot(gs[1, 0])
counts_month_location = df.groupby(['location', 'month']).size().unstack(fill_value=0)

if locations_to_plot:
    counts_month_location_filtered = counts_month_location.loc[locations_to_plot]
    sns.heatmap(counts_month_location_filtered, annot=True, fmt='d', cmap='summer', ax=ax2, linewidths=.5, linecolor='black')
else:
    sns.heatmap(counts_month_location, annot=True, fmt='d', cmap='summer', ax=ax2, linewidths=.5, linecolor='black')

ax2.set_title('Number of speeches per month and location')
ax2.set_xlabel('Month')
ax2.set_ylabel('Location')
ax2.tick_params(axis='x', rotation=45)
ax2.tick_params(axis='y', rotation=0)

# Year, Month, and Location (Top 5 Locations, grouped by year)
ax3 = fig.add_subplot(gs[1, 1])

# Get top 5 locations based on total speech count
top_5_locations = df['location'].value_counts().nlargest(5).index.tolist()

if top_5_locations:
    df_top_locations = df[df['location'].isin(top_5_locations)]

    # Group by year and location for this plot
    plot_data_ax3 = df_top_locations.groupby(['year', 'location']).size().unstack(fill_value=0)
    plot_data_ax3.plot(kind='bar', ax=ax3, rot=0, cmap='summer')
    ax3.set_title(f'Annual distribution of the top {len(top_5_locations)} locations')
    ax3.set_xlabel('Year')
    ax3.set_ylabel('Number of speeches')
    ax3.legend(title='Location', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax3.grid(axis='y', linestyle='--', alpha=0.7)
else:
    ax3.set_title('Nessuna location significativa per il 4° plot')
    ax3.text(0.5, 0.5, 'Non ci sono abbastanza dati per le top locations', horizontalalignment='center', verticalalignment='center', transform=ax3.transAxes)

plt.tight_layout()
plt.show()

## **TERM COUNT AND LEXICAL DIVERSITY**

## Term Count and Lexical Diversity

Before applying topic modeling, this section provides a quantitative overview of the corpus at the lexical level.  
The goal is to describe the internal structure of the speeches in terms of length, vocabulary size, and lexical diversity.

Four main measures are computed:

- **Tokens**: the total number of word occurrences in each speech. This measure is used to estimate the length of each document.
- **Types**: the number of unique word forms in each speech. This provides a first indication of vocabulary variety.
- **Type-Token Ratio (TTR)**: the ratio between types and tokens. It is used as a simple measure of lexical diversity, although it is sensitive to document length.
- **Entropy**: a measure of lexical distribution. Higher entropy suggests a more varied distribution of words, while lower entropy indicates a more repetitive vocabulary.

These descriptive statistics are useful for identifying differences in speech length and vocabulary usage across the corpus.  
They also help assess whether the dataset contains strong imbalances that may influence later stages of the analysis, such as preprocessing, embeddings, clustering, and topic modeling.

In [ ]:
data = df

documents = df['text'].tolist()
labels = df.columns.tolist()
label_names = df.columns.tolist()
doc_names = df['filename'].tolist()


print(f"Number of documents: {len(documents)}")
print(f"Number of labels: {len(labels)}")
print(f"Label names: {label_names}")
print(f"Document names: {doc_names[:5]}")

# a. Types

In [ ]:
for i in range(len(documents)):
  types = set(documents[i].split())
  print(f'Types of speech {i+1}: {len(types)}')

all_types = [len(set(text.split())) for text in df['text'].tolist()]
df['types_beforepp'] = all_types


# b. Tokens

In [ ]:
#wst is the "white space tokenizer" which is a very simple way of tokenizing, splitting tokenss where there is a space

def wst (text):
    return text.split()

In [ ]:
for i in range(len(documents)):
  tokens = wst(documents[i])
  print(f'Tokens of speech {i+1}: {len(tokens)}')

all_tokens = df['text'].apply(lambda x: len(wst(x)))
df['tokens_beforepp'] = all_tokens

In [ ]:
#distribution of types and tokens

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

all_types_mean = np.mean(all_types)
all_types_median = np.median(all_types)


all_tokens_mean = np.mean(all_tokens)
all_tokens_median = np.median(all_tokens)

# Create a figure and a GridSpec layout
fig = plt.figure(figsize=(15, 15))
gs = gridspec.GridSpec(2, 2, figure=fig, height_ratios=[1, 1.5])

# Plot for 'types'
ax0 = fig.add_subplot(gs[0, 0])
ax0.hist(all_types, bins=30, color='lightgreen', edgecolor='purple', linewidth=1.2)
ax0.set_title('Distribution of Types')
ax0.set_xlabel('Number of Types')
ax0.set_ylabel('Number of Speeches')
ax0.vlines(all_types_mean, 0, ax0.get_ylim()[1], color="red", linestyle='--', label="Mean")
ax0.vlines(all_types_median, 0, ax0.get_ylim()[1], color="orange", linestyle=':', label="Median")
ax0.legend()
ax0.grid(True, which='both', linestyle='--', linewidth=0.5)
ax0.tick_params(axis='x', rotation=45)

# Plot for 'tokens' (total words per document)
ax1 = fig.add_subplot(gs[0, 1])
ax1.hist(all_tokens, bins=30, color='lightgreen', edgecolor='purple', linewidth=1.2)
ax1.set_title('Distribution of Tokens')
ax1.set_xlabel('Number of Tokens')
ax1.set_ylabel('Number of Speeches')
ax1.vlines(all_tokens_mean, 0, ax1.get_ylim()[1], color="red", linestyle='--', label="Mean")
ax1.vlines(all_tokens_median, 0, ax1.get_ylim()[1], color="orange", linestyle=':', label="Median")
ax1.legend()
ax1.grid(True, which='both', linestyle='--', linewidth=0.5)
ax1.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
#number of tokens per speech

media = np.mean(all_tokens)
plt.plot(all_tokens, color= 'lightgreen', linewidth= 4, marker = 'o', markersize =6, markerfacecolor= 'purple')
plt.title('Tokens per speech')
plt.xlabel('Speeches')
plt.ylabel('Number of tokens')
plt.grid(True, linestyle= '--', linewidth=0.2)
plt.axhline(y=media, color='lightblue')

In [ ]:
df

# c. Type - Token Ratio

In [ ]:
#the Type-Token Ratio is defined as the number of unique word forms divided by the total number of word tokens. It provides a simple measure of lexical diversity.

def ttr (types,tokens):
    return len(types)/len(tokens)

#ttr of each speech
ttr_values = [ttr(set(text.split()), text.split()) for text in df['text'].tolist()]

for i in range(len(documents)):
  ttr_speech = ttr_values[i]
  print(f'TTr of speech {i+1}: {ttr_speech}')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

all_types = [len(set(text.split())) for text in df['text'].tolist()]
all_tokens = df['text'].apply(lambda x: len(wst(x))).tolist()
ttr_values = [len(set(text.split())) / len(wst(text)) for text in df['text'].tolist()]

# Create a figure and a single subplot
fig, ax1 = plt.subplots(figsize=(18, 10))

# Plot for Tokens and Types (Bar Chart) on the primary y-axis
x_indices = np.arange(len(all_tokens))
bar_width = 0.35

# Plot tokens
ax1.bar(x_indices - bar_width/2, all_tokens, bar_width, label='Number of Tokens', color='lightgreen', alpha=0.8)

# Plot types with a more 'blurred' appearance
ax1.bar(x_indices + bar_width/2, all_types, bar_width, label='Number of Types (Unique Words)', color='purple', alpha=0.5)

ax1.set_xlabel('Speech Index / Filename')
ax1.set_ylabel('Count (Tokens/Types)', color='purple')
ax1.tick_params(axis='y', labelcolor='black')
ax1.grid(True, linestyle='--', alpha=0.7)
ax1.set_xticks(x_indices)
ax1.set_xticklabels(df['filename'], rotation=90, fontsize=8)

# Create a secondary y-axis for TTR
ax2 = ax1.twinx()
ax2.plot(x_indices, ttr_values, label='Type-Token Ratio (TTR)', color='red', marker='o', linestyle='-', linewidth=2, markersize=5)

ax2.set_ylabel('Type-Token Ratio (TTR)', color='red')
ax2.tick_params(axis='y', labelcolor='red')
ax2.set_ylim(0, 1) # TTR typically ranges from 0 to 1

# Combine legends from both axes
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1+h2, l1+l2, loc='upper left')

ax1.set_title('Tokens, Types, and TT Ratio per Speech')

plt.tight_layout()
plt.show()

# d. Entropy

In [ ]:
# entropy measures how evenly word frequencies are distributed in a document. higher entropy indicates a more balanced lexical distribution, while lower entropy indicates stronger repetition of specific words.

def entropy(tokens):
    counter = Counter(tokens)
    total = len(tokens)

    probs = [count / total for count in counter.values()]

    return -sum(p * np.log2(p) for p in probs if p > 0)

In [ ]:
entropies = np.array([
    entropy(doc)
    for doc in df['text']
    if len(doc) > 0
])

print(f"Mean entropy: {entropies.mean():.4f}")
print(f"Std entropy: {entropies.std():.4f}")

import matplotlib.pyplot as plt

plt.hist(entropies, bins=50, color = 'lightgreen', edgecolor='purple')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.axvline(entropies.mean(), color='blue')
plt.xlabel("Entropy")
plt.ylabel("Number of documents")
plt.title("Entropy Distribution per Document")
plt.show()



from collections import defaultdict

entropy_by_year = defaultdict(list)

for doc, year in zip(df['text'], df["year"]):
    if len(doc) == 0:
        continue
    entropy_by_year[year].append(entropy(doc))

mean_entropy = {
    year: np.mean(vals)
    for year, vals in entropy_by_year.items()
}

for year, val in sorted(mean_entropy.items()):
    print(f"Mean entropy for {year}: {val:.3f}")


years = sorted(entropy_by_year.keys())
values = [entropy_by_year[y] for y in years]

plt.figure(figsize=(8, 5))

box = plt.boxplot(
    values,
    labels=years,
    patch_artist=True,
    widths=0.6
)


colors = ["#39FF14", "#C43DFF"]

for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)


for median in box['medians']:
    median.set(color='black', linewidth=2)

# outliers
for flier in box['fliers']:
    flier.set(marker='P', color='purple', alpha=1)

plt.xlabel("Year", fontsize=12)
plt.ylabel("Entropy", fontsize=12)
plt.title("Entropy per Year", fontsize=14, fontweight='bold')

plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.show()


##**PREPROCESSING**

# a. Preprocessing

In [ ]:
from nltk import word_tokenize
import nltk
from string import punctuation
punctuation += '$£&'
from nltk.corpus import stopwords
EN_STOPWORDS = set(stopwords.words('english'))
from collections import Counter


# Additional tokens commonly produced by tokenization or transcription formatting. These elements carry little semantic meaning and are removed from the corpus.
extra_noise = {
    "'s", "n't", "'re", "'ve", "'ll", "'d", "'m",
    "``", "''", "’", "“", "”", "—", "–"}


# Define a reusable preprocessing function so that the same cleaning procedure is applied consistently to every speech in the corpus.

def process_text_pipeline(text):
    # Tokenize
    tok = word_tokenize(text)

    # Remove punctuation
    tok_nopunct = [i for i in tok if i not in punctuation]

    # Lowercase
    lower_tokens = [t.lower() for t in tok_nopunct]

    # Remove NLTK stopwords
    clean_tokens = [t for t in lower_tokens if t not in EN_STOPWORDS]

    # Remove extra noise
    clean_extra = [t for t in clean_tokens if t not in extra_noise]

    #Remove numbers
    clean_numbers = [t for t in clean_extra if not t.isdigit()]

    return clean_numbers


# Apply the pipeline to all texts in the DataFrame
df['cleaned_text'] = df['text'].apply(process_text_pipeline)

df['len_pptext'] = df['cleaned_text'].apply(len)

print("First cleaned text example:")
print(df['cleaned_text'].iloc[0])
print(f"Length of first cleaned text: {len(df['cleaned_text'].iloc[0])} tokens")

In [ ]:
# Each value in cleaned_text is a list of cleaned word tokens.
df['cleaned_text'][0]

In [ ]:
# Convert each list of cleaned tokens back into a single string. This format is required by vectorizers and embedding models.
documents_string = data["cleaned_text"].apply(lambda tokens: " ".join(tokens)).tolist()

In [ ]:
documents_string[0]

# b. Comparing Raw and Cleaned Text Length Distributions


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(12, 7))

# Calculate histograms for both distributions
hist_before, bins_before = np.histogram(all_tokens, bins='auto', density=True)
hist_after, bins_after = np.histogram(df['len_pptext'], bins='auto', density=True)

# Calculate bin centers for plotting
bin_centers_before = (bins_before[:-1] + bins_before[1:]) / 2
bin_centers_after = (bins_after[:-1] + bins_after[1:]) / 2

# Plot the line for raw text length distribution
plt.plot(bin_centers_before, hist_before, color='lightgreen', linewidth=2, label='Before Preprocessing')

# Plot the line for cleaned text length distribution
plt.plot(bin_centers_after, hist_after, color='purple', linewidth=2, label='After Preprocessing', linestyle='--')

plt.xlabel('Number of Tokens')
plt.ylabel('Density')
plt.title('Overlay of Token Distribution Before and After Preprocessing (Line Plot)')
plt.legend()
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get the number of speeches for the x-axis
speech_indices = np.arange(len(df))

plt.figure(figsize=(18, 8))

# Plot tokens before preprocessing
plt.plot(speech_indices,
         all_tokens,
         color='purple',
         linewidth=2,
         marker='o',
         markersize=6,
         markerfacecolor='red',
         label='Tokens Before Preprocessing')

# Plot tokens after preprocessing
plt.plot(speech_indices,
         df['len_pptext'],
         color='lightgreen',
         linewidth=2,
         marker='s',
         markersize=6,
         markerfacecolor='blue',
         label='Tokens After Preprocessing')

plt.title('Comparison of Token Counts Before and After Preprocessing Per Speech')
plt.xlabel('Speech Index')
plt.ylabel('Number of Tokens')
plt.xticks(speech_indices, df['filename'], rotation=90, fontsize=8)
plt.grid(True, linestyle='--', linewidth=0.2)
plt.legend()
plt.tight_layout()
plt.show()

# c. Counter

In [ ]:
# Flatten the list of lists of cleaned tokens into a single list of tokens
all_cleaned_tokens = [token for sublist in df['cleaned_text'] for token in sublist]
vocab_counter = Counter(all_cleaned_tokens)
vocab_counter.most_common(30)

In [ ]:
from wordcloud import WordCloud
from collections import Counter
import matplotlib.pyplot as plt

# WORD CLOUD - frequency: words with highest frequency


counter_2019 = Counter()
counter_2020 = Counter()

for doc, year in zip(df["cleaned_text"], df["year"]):
    if year == 2019:
        counter_2019.update(doc)
    elif year == 2020:
        counter_2020.update(doc)


# Word Cloud 2019
wc_2019 = WordCloud(
    width=1000,
    height=500,
    background_color="white",
    max_words=30,
    collocations=False
).generate_from_frequencies(counter_2019)

plt.figure(figsize=(12, 6))
plt.imshow(wc_2019, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud - 2019", fontsize=18)
plt.show()


# Word Cloud 2020
wc_2020 = WordCloud(
    width=1000,
    height=500,
    background_color="white",
    max_words=30,
    collocations=False
).generate_from_frequencies(counter_2020)

plt.figure(figsize=(12, 6))
plt.imshow(wc_2020, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud - 2020", fontsize=18)
plt.show()

## **CHUNKS**

The speeches in the corpus are relatively long and often contain multiple themes within the same document.  If each full speech were passed to BERTopic as a single document, the embedding model would produce one vector representation for the entire speech.  This vector would summarize all the themes in the speech into a single semantic representation, potentially making it harder for the topic model to identify more specific and coherent topics.

To address this issue, each preprocessed speech is divided into smaller overlapping chunks.  
Each chunk contains a fixed number of words and partially overlaps with the previous one.  
The overlap helps preserve semantic continuity between adjacent chunks and reduces the risk of cutting a relevant passage too abruptly.

In this project, a chunk size of 200 words and an overlap of 50 words are used.  
This means that each new chunk starts 150 words after the previous one.  
Chunks shorter than 80 words are excluded to avoid creating very small text units with limited semantic information.

The resulting chunks are used as the input documents for embedding computation and topic modeling, while metadata are preserved to link each chunk back to its original speech, year, month, and location.

In [ ]:
# Convert filenames from a series to list

filenames = data["filename"].tolist()

print(type(filenames))
print(len(filenames))
print(filenames[:5])

In [ ]:
#create chunks

chunked_docs = []
chunk_metadata = []

chunk_size = 200      #max 200 words
overlap = 50          #overlap of 50
step = chunk_size - overlap

global_chunk_id_counter = 0

for idx, row in data.iterrows():
    words = row["cleaned_text"]

    for i in range(0, len(words), step):
        chunk_words = words[i:i+chunk_size]

        if len(chunk_words) >= 80:
            chunk = " ".join(chunk_words)

            chunked_docs.append(chunk)
            chunk_metadata.append({
                "doc_id": idx,
                "filename": row["filename"],
                "year": row["year"],
                "month": row["month"],
                "location": row["location"],
                "chunk_id": global_chunk_id_counter,
                "start_word": i,
                "end_word": i + len(chunk_words),
                "n_words": len(chunk_words)
            })
            global_chunk_id_counter += 1

print("Number of original speeches:", len(data))
print("Number of created chunks:", len(chunked_docs))
print(chunked_docs[0][:1000])
print(chunk_metadata[0])

In [ ]:
# assign each chunk with their id to the speech they belong to
chunk_ids_per_doc = {}
for entry in chunk_metadata:
    doc_id = entry['doc_id']
    chunk_id = entry['chunk_id']
    if doc_id not in chunk_ids_per_doc:
        chunk_ids_per_doc[doc_id] = []
    chunk_ids_per_doc[doc_id].append(chunk_id)

df['chunk_ids'] = df.index.map(chunk_ids_per_doc)

# Print full chunk_ids for the first two documents to demonstrate global uniqueness
print(f"Chunk IDs for Document 0 ({df['filename'].iloc[0]}):")
print(df['chunk_ids'].iloc[0])
print("\n")
print(f"Chunk IDs for Document 1 ({df['filename'].iloc[1]}):")
print(df['chunk_ids'].iloc[1])

In [ ]:
#save chunks data

chunks_df = pd.DataFrame(chunk_metadata)
chunks_df["text"] = chunked_docs

chunks_df.to_csv(
    get_table_path("chunks_metadata.csv"),
    index=False
)

with open(get_table_path("chunked_docs.json"), "w", encoding="utf-8") as f:
    json.dump(chunked_docs, f, ensure_ascii=False, indent=2)

print("Chunks and metadata saved.")

In [ ]:
#show the df with the new column

df

## **MANUAL BERTOPIC**

This section implements a manual version of the BERTopic pipeline.
Instead of relying entirely on the automatic BERTopic class, the main steps of the model are reproduced separately in order to make the topic modeling process more transparent and interpretable.

The manual pipeline includes the following stages:

1. *Sentence-BERT embeddings*: each chunk is transformed into a dense semantic vector using the all-MiniLM-L6-v2 model.
2. *Dimensionality reduction with UMAP*: high-dimensional embeddings are projected into a lower-dimensional space.
3. *Clustering with HDBSCAN*: semantically similar chunks are grouped into clusters, while noisy or ambiguous chunks are assigned to the outlier class.
4. *CountVectorizer*: removal of extra words very frequent but not informative; this will make the topic representation more efficient.
5. *Topic representation with c-TF-IDF*: each cluster is represented through its most distinctive words.
6. *Topic visualization*: the resulting topics are visualized in two-dimensional and three-dimensional semantic spaces.
7. *Evaluation*: the quality of the manually generated topics is assessed using coherence and topic diversity.

This manual implementation makes it possible to inspect each component of the BERTopic workflow separately.
It also allows more direct control over the parameters used for embedding, dimensionality reduction, clustering, and topic representation.

# *1. SBert*


The first step of the manual BERTopic pipeline is the computation of semantic embeddings.  
Each chunk is encoded using the `all-MiniLM-L6-v2` Sentence-BERT model, which maps text segments into dense vector representations.

These embeddings capture semantic similarity between chunks and are used as the input for the following stages of dimensionality reduction and clustering.  
Since embedding computation can be time-consuming, the resulting vectors are saved to disk.  
If the embedding file already exists, the notebook loads the pre-computed embeddings instead of recomputing them.

In [ ]:
from sentence_transformers import SentenceTransformer

sbert_model_name = "all-MiniLM-L6-v2"
embedding_path = get_embedding_path("embeddings_manual.npy")

print(f"Checking for embedding file at: {embedding_path}")
print(f"File exists: {os.path.exists(embedding_path)}")

if os.path.exists(embedding_path):
    print("Loading pre-computed SBERT embeddings for manual BERTopic...")
    embedding = np.load(embedding_path)
else:
    print("Computing SBERT embeddings for manual BERTopic...")
    model = SentenceTransformer(sbert_model_name)
    embedding = model.encode(
        chunked_docs,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    np.save(embedding_path, embedding)
    print(f"Embeddings saved in: {embedding_path}")

print("Embedding shape:", embedding.shape)

In [ ]:
similarity = cosine_similarity(embedding)

In [ ]:
n = 20

small_matrix = similarity[:n, :n]

plt.figure(figsize=(12, 10))

plt.imshow(small_matrix, cmap='viridis')
plt.colorbar(label='Cosine similarity')

plt.xticks(
    ticks=range(n),
    rotation=90,
    fontsize=8
)

plt.yticks(
    ticks=range(n),
    fontsize=8
)

plt.title(f'Semantic similarity in the first {n} chunks')
plt.xlabel('Chunks')
plt.ylabel('Chunks')

plt.tight_layout()
plt.show()

# *2. UMAP: dimensionality reduction*

The SBERT embeddings are high-dimensional vectors.  
Before clustering, UMAP is used to reduce the dimensionality of the embedding space while preserving local semantic relationships between chunks.

This step is useful because clustering algorithms generally perform better on lower-dimensional representations.  
The reduced embeddings are then passed to HDBSCAN for density-based clustering.

In [ ]:
# n_neighbors controls how much local versus global structure is preserved.
# n_components defines the dimensionality of the reduced embedding space.
# metric="cosine" is suitable for semantic embeddings.
# random_state is fixed to improve reproducibility.

reducer = umap.UMAP(n_neighbors=15, n_components=30, min_dist=0.0, metric="cosine", random_state=0, )
reduced_embeddings = reducer.fit_transform(embedding)

In [ ]:
pd.DataFrame(reduced_embeddings)

# *3. Clustering HDBSCAN*

After dimensionality reduction, HDBSCAN is used to group semantically similar chunks into clusters.  
Unlike algorithms that require a predefined number of clusters, HDBSCAN identifies clusters based on density.

This is particularly useful for political speech data, where the number of topics is not known in advance.  
Chunks that do not clearly belong to any cluster are assigned to the outlier category `-1`.

In [ ]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=10,         #determines the smallest grouping considered a valid, final cluster
    min_samples = 1,            #It dictates how many neighbors a point needs to have within its neighborhood to be considered a "core point".
    metric="euclidean",
    cluster_selection_method="eom"
)

clusters = clusterer.fit_predict(reduced_embeddings)

In [ ]:
clusters

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

clusters_array = np.array(clusters)

clustered_mask = clusters_array != -1
noise_mask = clusters_array == -1

clustered_points = reduced_embeddings[clustered_mask]
clustered_clusters = clusters_array[clustered_mask]
noise_points = reduced_embeddings[noise_mask]

plt.figure(figsize=(12, 9))

# Noise
plt.scatter(
    noise_points[:, 0],
    noise_points[:, 1],
    c="lightgray",
    s=30,
    alpha=0.3,
    label="Noise points",
    zorder=1
)

# Cluster
scatter = plt.scatter(
    clustered_points[:, 0],
    clustered_points[:, 1],
    c=clustered_clusters,
    cmap="Set2",
    s=75,
    alpha=0.85,
    edgecolors="white",
    linewidths=0.4,
    zorder=2
)


unique_clusters = np.unique(clustered_clusters)

for cluster_id in unique_clusters:
    points = reduced_embeddings[clusters_array == cluster_id]
    center_x = points[:, 0].mean()
    center_y = points[:, 1].mean()

    plt.text(
        center_x,
        center_y,
        str(cluster_id),
        fontsize=11,
        fontweight="bold",
        ha="center",
        va="center",
        bbox=dict(
            facecolor="white",
            edgecolor="black",
            boxstyle="round,pad=0.25",
            alpha=0.75
        ),
        zorder=3
    )

plt.title(
    "HDBSCAN clusters on UMAP projection",
    fontsize=18,
    fontweight="bold",
    pad=15
)

plt.xlabel("UMAP dimension 1", fontsize=12)
plt.ylabel("UMAP dimension 2", fontsize=12)

plt.grid(True, linestyle="--", alpha=0.22)

cbar = plt.colorbar(scatter)
cbar.set_label("Cluster ID", fontsize=11)

plt.legend(frameon=True, loc="best")

plt.tight_layout()
plt.show()

In [ ]:
#representation of speeches and their chunks

last_printed_doc_id = None

for idx, (doc, c) in enumerate(zip(chunked_docs, clusters)):
    current_metadata = chunk_metadata[idx]
    current_doc_id = current_metadata['doc_id']
    current_chunk_id = current_metadata['chunk_id']

    # Check if a new document has started
    if current_doc_id != last_printed_doc_id:
        print(f"\n--- Document: {current_metadata['filename']} ---")
        last_printed_doc_id = current_doc_id

    print(f"{c}: {doc}")

    # Get all chunk_ids for the current document
    document_chunk_ids = df['chunk_ids'].iloc[current_doc_id]

    # Check if this is the last chunk for the document
    if current_chunk_id == document_chunk_ids[-1]:
        print("\n" + "-" * 50 + "\n")

# *4. CountVectorizer*

CountVectorizer determines which words are included in the final vocabulary. By setting `stop_words=stopwords`, words with little informational value are removed; with `ngram_range=(1,2)`, both single words and pairs of consecutive words are considered; and with `min_df=3`, only terms that appear in at least three chunks are retained.

This step is especially important because, as shown in the previous term-count section, some words are extremely frequent in the corpus but not very informative for topic interpretation. If these words are kept, they tend to appear across many different topics and make the distinction between topics less clear. In other words, they introduce noise into the topic representations.

For this reason, words such as `thank`, `great`, and `people` are removed at this stage. This was not done earlier, during preprocessing, because removing them before computing the SBERT embeddings could have altered the semantic representation of the chunks. Instead, the words are removed only before the c-TF-IDF step, so that they cannot be selected as representative terms for the topics.

This configuration directly affects c-TF-IDF, because c-TF-IDF can only assign weights to the words that CountVectorizer passes to it. As a result, c-TF-IDF is encouraged to highlight more specific and informative terms, making the final topic labels clearer and more meaningful.


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

trump_stopwords = [
    "thank", "thanks", "everybody", "hello",
    "good", "really", "very", "look", "va",
    "people", "country", "know", "like", "way",
    "sir", "said", "say", "tell", "get", "got",
    "going", "think", "know", "want", "guy", "okay",
    "lot", "many", "much", "would", "like", "come",
    "years", "year", "today", "tonight", "winning", "won"
]

stopwords = list(ENGLISH_STOP_WORDS) + trump_stopwords

vectorizer = CountVectorizer(
    stop_words=stopwords,
    ngram_range=(1, 2),
    min_df=3
)

# *5. Topic representation with c-tf-idf*

After clustering, each cluster is treated as a single aggregated document composed of all chunks assigned to that cluster.  
A class-based TF-IDF representation is then computed to identify the most distinctive words for each topic.

This step makes the clusters interpretable: while HDBSCAN groups chunks based on semantic similarity, c-TF-IDF extracts the terms that best describe each cluster.   

c-TF-IDF identifies the most representative terms of each topic by giving higher weight to words that are frequent within a specific topic but less common across the other topics.

The resulting top words are used to manually label and interpret the topics.

In [ ]:
# Group chunks by their assigned cluster. Outliers labeled as -1 are excluded from topic representation

cluster_docs = {}

for doc, c in zip(chunked_docs, clusters):
    if c == -1:
        continue
    cluster_docs.setdefault(c, []).append(doc)
cluster_docs

In [ ]:
# Store topic IDs and concatenate all chunks belonging to the same topic.
topic_ids = list(cluster_docs.keys())
cluster_texts = [" ".join(cluster_docs[topic_id]) for topic_id in topic_ids]

In [ ]:
# Convert cluster-level documents into a term-frequency matrix.
X_counts = vectorizer.fit_transform(cluster_texts)

terms = vectorizer.get_feature_names_out()

In [ ]:
#calculate c-tf-idf
from bertopic.vectorizers import ClassTfidfTransformer
ctfidf_model = ClassTfidfTransformer()
X_ctfidf = ctfidf_model.fit_transform(X_counts)

ctfidf = X_ctfidf.toarray()

In [ ]:
n_top = 6

sorted_topic_ids = sorted(topic_ids)

for topic_id in sorted_topic_ids:
    idx = topic_ids.index(topic_id)
    row = ctfidf[idx]
    top = np.argsort(row)[::-1][:n_top]

    print(f"\nTopic {topic_id}")
    for i in top:
        print(terms[i], f"{row[i]:.3f}")

# 3D Visualization

To visually inspect the topic structure, the SBERT embeddings are projected into a three-dimensional UMAP space.  Each point corresponds to a chunk, and colors represent the topic assigned by the manual clustering pipeline.

This visualization helps assess whether the clusters occupy distinguishable regions in the semantic space.  It also allows interactive inspection of the chunks through filename and text previews.

In [ ]:
#reduce to 3 dimensions so that a 3d visualizatioon is possible

reducer = umap.UMAP(n_neighbors=3, n_components=3, min_dist=0.0, metric="cosine", random_state=0, )
reduced_embeddings = reducer.fit_transform(embedding)

In [ ]:
!pip install plotly

In [ ]:
from umap import UMAP
import plotly.express as px

umap_model_visual = UMAP(
    n_neighbors=15,
    n_components=3,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)
embeddings_3d = umap_model_visual.fit_transform(embedding)

# Create a DataFrame from chunk_metadata
df_chunks_for_plot = pd.DataFrame(chunk_metadata)

# Add the UMAP embeddings and clusters to this new DataFrame
df_chunks_for_plot['x'] = embeddings_3d[:, 0]
df_chunks_for_plot['y'] = embeddings_3d[:, 1]
df_chunks_for_plot['z'] = embeddings_3d[:, 2]
df_chunks_for_plot['topic'] = clusters

# Add a text preview from the chunked_docs themselves
df_chunks_for_plot['text_preview'] = [doc[:300] for doc in chunked_docs]

fig = px.scatter_3d(
    df_chunks_for_plot,
    x='x',
    y='y',
    z='z',
    color='topic',
    hover_name='filename',
    hover_data={
        'text_preview': True,
        'x': False,
        'y': False,
        'z': False
    },
    title='Mappa 3D degli embeddings SBERT colorata per topic BERTopic (basata su chunks)'
)

fig.update_traces(
    marker=dict(size=6, opacity=0.85)
)

fig.update_layout(
    width=1000,
    height=780,
    scene=dict(
        xaxis_title='UMAP 1',
        yaxis_title='UMAP 2',
        zaxis_title='UMAP 3'
    ),
    legend_title_text='Topic'
)

fig.show()

# *6. Evaluation*

The manual BERTopic results are evaluated using two complementary metrics: coherence and topic diversity.

**Coherence** measures the semantic consistency of the top words within each topic. A higher coherence score suggests that the words representing a topic tend to occur in similar contexts and are more interpretable.

**Topic diversity** measures the proportion of unique words across all topic representations.  A higher diversity score indicates that different topics are represented by more distinct vocabularies, reducing redundancy between topics.

Together, these metrics provide a quantitative assessment of topic quality, although final interpretation still requires qualitative inspection of the topic keywords and representative chunks.

In [ ]:
!pip install gensim
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

In [ ]:

def compute_coherence(topics, texts: list[list[str]]):
    dictionary = Dictionary(texts)
    cm = CoherenceModel(
        topics=topics,
        texts=texts,
        dictionary=dictionary,
        coherence='c_v'
    )
    return cm.get_coherence()

def topic_diversity(topics):
    top_words = set(word for topic in topics for word in topic)
    total_words = len(topics) * len(topics[0])
    return len(top_words) / total_words


# 1. Prepare topic words for manual BERTopic coherence
manual_bertopic_topics_for_coherence = []

for idx, row in enumerate(ctfidf):
    current_topic_id = topic_ids[idx]
    top_word_indices = np.argsort(row)[::-1][:10] # Top 10 words, as used for coherence
    topic_words = [terms[i] for i in top_word_indices]
    manual_bertopic_topics_for_coherence.append(topic_words)

# 2. Prepare tokenized documents for manual BERTopic coherence
manual_bertopic_texts_tokenized = [chunk.split() for chunk in chunked_docs]

# 3. Calculate coherence for manual BERTopic
manual_bertopic_coh = compute_coherence(manual_bertopic_topics_for_coherence, manual_bertopic_texts_tokenized)

print(f"Coherence for Manual BERTopic: {manual_bertopic_coh}")

In [ ]:
manual_bertopic_div = topic_diversity(manual_bertopic_topics_for_coherence)
print(f"Diversity for Manual BERTopic: {manual_bertopic_div:.3f}")

## **Automatic Bertopic**

In this section, BERTopic is applied in its automatic configuration to identify the main latent topics within the corpus of Trump speeches. The model is run on preprocessed and chunked documents and combines the standard BERTopic pipeline: Sentence-BERT embeddings, UMAP dimensionality reduction, HDBSCAN clustering, and c-TF-IDF keyword extraction.

At this exploratory stage, the model is allowed to discover topics automatically, without imposing a fixed number of clusters or predefined labels. This makes it useful for obtaining an initial map of the corpus and detecting broad recurring themes, such as immigration, trade, the economy, foreign policy, the military, healthcare, and attacks on political opponents.

Each topic is interpreted through both its c-TF-IDF keywords and its representative chunks. This is important because topic labels should not be treated as fixed or definitive categories, but as data-driven summaries of recurrent linguistic and rhetorical patterns. Individual keywords can be ambiguous, while the surrounding text helps clarify the actual context in which they are used.

The outlier topic `-1` contains chunks that HDBSCAN does not assign to any stable cluster. In this corpus, this residual category is especially relevant because political rallies often include digressions, repeated slogans, audience interaction, applause formulas, jokes, and generic campaign rhetoric. Therefore, topic `-1` should not be interpreted as a coherent political theme, but rather as a group of noisy or less semantically consistent segments.

Overall, the automatic BERTopic model provides a useful exploratory baseline. It highlights the main thematic areas of the speeches, but some topics may still overlap or include generic rhetorical terms, which limits their interpretability. For this reason, the automatic results are not treated as the final topic structure, but as a first step toward refinement through additional stopword removal, manual topic labeling, and comparison with temporal or geographical metadata.


In [ ]:
from bertopic import BERTopic

In [ ]:
import random
import numpy as np
import torch

#to block global casuality. this controls the main sources of randomness used by Python, NumPy and PyTorch. reproducibility is important because BERTopic uses stochastic components such as UMAP.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from umap import UMAP
from hdbscan import HDBSCAN

#custom stopwords specific to Trump rally speeches.
trump_stopwords = [
    "thank", "thanks", "everybody", "hello",
    "good", "really", "very", "look", "va",
    "people", "country", "know", "like", "way",
    "sir", "said", "say", "tell", "get", "got",
    "going", "think", "know", "want", "guy", "okay",
    "lot", "many", "much", "would", "like", "come", "made", "making",
    "years", "year", "today", "tonight", "winning", "won",  "bad", "best", "better",
    "never", "ever",
    "back", "take",
    "done", "doing",
    "also", "even",
    "let", "lets",
    "place", "places",
]

stopwords = list(ENGLISH_STOP_WORDS) + trump_stopwords

vectorizer_model = CountVectorizer(
    stop_words=stopwords,
    ngram_range=(1, 2),
    min_df=3
)

# random_state=SEED improves reproducibility, while n_jobs=1 avoids parallel randomness.

umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=SEED,
    n_jobs=1
)

#min_cluster_size=10 avoids creating very small topics.
#prediction_data=True is useful for later topic prediction or probability estimation.
#gen_min_span_tree=True allows additional inspection of the clustering structure if needed.
hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True,
    gen_min_span_tree=True
)

In [ ]:
#model and save embedding
from sentence_transformers import SentenceTransformer

sbert_model_name = "all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(sbert_model_name)

embedding_path_auto = get_embedding_path("embeddings_bertopic_auto.npy")

print(f"Checking for embedding file at: {embedding_path_auto}")
print(f"File exists: {os.path.exists(embedding_path_auto)}")

if os.path.exists(embedding_path_auto):
    print("Loading pre-computed embeddings for automatic BERTopic...")
    embeddings_auto = np.load(embedding_path_auto)
else:
    print("Computing embeddings for automatic BERTopic...")
    embeddings_auto = embedding_model.encode(
        chunked_docs,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    np.save(embedding_path_auto, embeddings_auto)
    print(f"Embeddings saved in: {embedding_path_auto}")

print("Embeddings shape:", embeddings_auto.shape)

In [ ]:
#create model from tthe customized componentss from before
topic_model = BERTopic(
    min_topic_size=10,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    embedding_model=embedding_model
)


In [ ]:
#fit the model with chunks
topics, probs = topic_model.fit_transform(chunked_docs, embeddings=embeddings_auto)

In [ ]:
# Save topic information
topic_info = topic_model.get_topic_info()
topic_info.to_csv(
    get_table_path("bertopic_topic_info.csv"),
    index=False
)

# Save topic assignment for each chunk
topic_assignments = pd.DataFrame(chunk_metadata)
topic_assignments["text"] = chunked_docs
topic_assignments["topic"] = topics

topic_assignments.to_csv(
    get_table_path("bertopic_topic_assignments.csv"),
    index=False
)

# Save probabilities if available
if probs is not None:
    np.save(
        get_embedding_path("bertopic_probabilities.npy"),
        probs
    )

# Save the BERTopic model
topic_model.save(
    get_model_path("bertopic_model")
)

print("BERTopic outputs saved.")

In [ ]:
topic_model.get_topic_info()



In [ ]:
topic_model.get_topic(7)

## **Evaluation**

The same Evaluation section as for Manual Bertopic is computed. Again, the metrics assesssed are coherence and diversity.

In [ ]:
import gensim
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

def compute_coherence(topics, texts: list[list[str]]):
    dictionary = Dictionary(texts)
    cm = CoherenceModel(
        topics=topics,
        texts=texts,
        dictionary=dictionary,
        coherence='c_v'
    )
    return cm.get_coherence()

common_tokenized_texts = [chunk.split() for chunk in chunked_docs]

def simple_tokenize(docs):
    tokenized = []
    for doc in docs:
        doc = doc.lower()
        doc = re.sub(r"[^a-z0-9\s]", "", doc)
        tokens = doc.split()
        tokenized.append(tokens)
    return tokenized

# Define bertopic_topics by extracting top words from the BERTopic model
bertopic_topics = []
for topic_id in sorted(topic_model.get_topics().keys()):
    if topic_id == -1:
        continue
    top_words_with_scores = topic_model.get_topic(topic_id)
    topic_words = [word for word, score in top_words_with_scores[:10]]
    bertopic_topics.append(topic_words)

bertopic_texts_tokenized = [chunk.split() for chunk in chunked_docs]

bertopic_coh = compute_coherence(bertopic_topics, bertopic_texts_tokenized)


In [ ]:
#coherence
bertopic_coh

In [ ]:
#diversity
bertopic_div = topic_diversity(bertopic_topics)
bertopic_div

In [ ]:
evaluation_results = {
    "model": "BERTopic automatic",
    "coherence_c_v": float(bertopic_coh),
    "topic_diversity": float(bertopic_div),
    "n_topics_excluding_outlier": len(bertopic_topics),
    "n_chunks": len(chunked_docs),
    "embedding_model": sbert_model_name,
    "chunk_size": chunk_size,
    "overlap": overlap
}

# Save as JSON
with open(get_table_path("evaluation_results.json"), "w", encoding="utf-8") as f:
    json.dump(evaluation_results, f, indent=4)

# Save as CSV
pd.DataFrame([evaluation_results]).to_csv(
    get_table_path("evaluation_results.csv"),
    index=False
)

print("Evaluation results saved.")

## **INTERPRETATION OF RESULTS**


The BERTopic visualizations help evaluate the quality and interpretability of the automatic topic model. The topic word barchart shows the most representative terms for each topic according to c-TF-IDF, making it easier to assign meaningful labels and identify topics that still contain generic or noisy vocabulary.

The intertopic distance map shows the semantic relationship between topics. Topics positioned close to each other share similar vocabulary or meaning, while distant topics are more clearly distinct. In political speeches, some overlap is expected because themes such as immigration, the economy, national security, and attacks on political opponents often appear together.

The hierarchical clustering graph shows whether different topics can be grouped into broader thematic families. This is useful for understanding whether apparently separate topics actually belong to the same macro-area, such as border security, economic policy, foreign policy, or campaign rhetoric.

The similarity heatmap provides a direct comparison between topics. High similarity values may indicate thematic overlap or redundancy, while lower similarity values suggest more independent and clearly separated topics.

Overall, these graphs show that the automatic BERTopic model provides a useful exploratory structure, but also that some topics require further manual interpretation. The visual results therefore support the next refinement step, where topic labels, stopwords, and thematic coherence are examined more carefully.

In [ ]:
!pip install -U plotly kaleido nbformat

In [ ]:
import plotly.io as pio

pio.renderers.default = "colab"

In [ ]:
fig = topic_model.visualize_barchart(top_n_topics=18, n_words=10)
fig.write_html(get_figure_path("bertopic_intertopic_bar.html"))
fig.show()

In [ ]:
fig = topic_model.visualize_topics()
fig.write_html(get_figure_path("bertopic_intertopic_map.html"))
fig.show()

In [ ]:
fig = topic_model.visualize_heatmap()
fig.write_html(get_figure_path("bertopic_similarity_heatmap.html"))
fig.show()

In [ ]:
fig = topic_model.visualize_hierarchy()
fig.write_html(get_figure_path("bertopic_hierarchy.html"))
fig.show()

In [ ]:
fig = topic_model.visualize_documents(chunked_docs, embeddings=embeddings_auto)
fig.write_html(get_figure_path("bertopic_documents_topics.html"))
fig.show()

In [ ]:
import plotly.express as px

topic_info = topic_model.get_topic_info()

topic_info_no_outliers = topic_info[topic_info["Topic"] != -1]

fig = px.bar(
    topic_info_no_outliers.head(15),
    x="Count",
    y="Name",
    orientation="h",
    title="Top 15 topics by number of chunks",
    labels={"Count": "Number of chunks", "Name": "Topic"}
)

fig.update_layout(
    yaxis={'categoryorder':'total ascending'},
    height=700
)

fig.write_html(get_figure_path("bertopic_top15_topics_by_chunks.html"))

fig.show()

In [ ]:
import plotly.express as px

# Get topic names from the BERTopic model
topic_info_df = topic_model.get_topic_info()
# Create a mapping from Topic ID to Topic Name
topic_name_mapping = topic_info_df.set_index('Topic')['Name'].to_dict()
# Map the numerical topic IDs in df_chunks_for_plot to their names
df_chunks_for_plot['topic_name'] = df_chunks_for_plot['topic'].map(topic_name_mapping).fillna('Noise')
# Ensure topic is categorical for distinct colors in plotly
df_chunks_for_plot['topic_name'] = df_chunks_for_plot['topic_name'].astype('category')

# Sort filenames for consistent plotting order on y-axis
sorted_filenames = df['filename'].unique().tolist()
df_chunks_for_plot['filename'] = pd.Categorical(df_chunks_for_plot['filename'], categories=sorted_filenames, ordered=True)

# Calculate topic percentages for each speech and prepare a summary for hover data
temp_df_topic_summary = df_chunks_for_plot.groupby(['filename', 'topic_name']).size().reset_index(name='count')
total_chunks_per_filename = df_chunks_for_plot.groupby('filename').size().reset_index(name='total_chunks')
temp_df_topic_summary = pd.merge(temp_df_topic_summary, total_chunks_per_filename, on='filename')
temp_df_topic_summary['percentage'] = (temp_df_topic_summary['count'] / temp_df_topic_summary['total_chunks']) * 100

def format_topics_for_speech(group):
    group = group.sort_values(by='percentage', ascending=False)
    summary_parts = []
    for _, row in group.iterrows():
        summary_parts.append(f"- {row['topic_name']}: {row['percentage']:.1f}% ({int(row['count'])} chunks)")
    return "Topics in this Speech:<br>" + "<br>".join(summary_parts)

speech_topic_summary = temp_df_topic_summary.groupby('filename').apply(format_topics_for_speech, include_groups=False).reset_index(name='speech_topic_summary')


# Drop 'speech_topic_summary' if it already exists in df_chunks_for_plot to prevent MergeError
if 'speech_topic_summary' in df_chunks_for_plot.columns:
    df_chunks_for_plot = df_chunks_for_plot.drop(columns=['speech_topic_summary'])
# --- FIX END ---

df_chunks_for_plot = pd.merge(df_chunks_for_plot, speech_topic_summary, on='filename', how='left')

# Create the interactive scatter plot
fig = px.scatter(
    df_chunks_for_plot,
    x='start_word',
    y='filename',
    color='topic_name', # Use topic names for coloring
    hover_data={
        'text_preview': True,
        'topic_name': True,
        'filename': True,
        'chunk_id': True,
        'start_word': True,
        'end_word': True,
        'speech_topic_summary': True # Add the new summary column to hover data
    },
    title='Topics per Chunk within Speeches',
    labels={
        "start_word": "Starting Word Index within Speech",
        "filename": "Speech Document",
        "topic_name": "Topic",
        "speech_topic_summary": "Topic Distribution in Speech"
    }
)

fig.update_layout(
    xaxis_title="Starting Word Index within Speech",
    yaxis_title="Speech Document",
    height=len(df['filename'].unique()) * 25 + 200, # Dynamic height based on number of speeches
    width=1200,
    showlegend=True,
    hovermode="closest",
    legend_title_text='Topic'
)

fig.update_traces(marker=dict(size=10, opacity=0.7, line=dict(width=0.5, color='DarkSlateGrey')))

fig.write_html(get_figure_path("bertopic_auto_speechchunks.html"))
fig.show()

In [ ]:
import plotly.express as px

# Standardize the name for the Noise topic (Topic ID -1)
# The topic_name_mapping from previous cells is used to get the original BERTopic generated name for topic -1.
# This ensures that all instances of topic ID -1, whether initially named by BERTopic or as 'Noise' by fillna,
# are consolidated under a single, user-friendly label.
original_noise_topic_generated_name = topic_name_mapping.get(-1, 'Noise') # Get the generated name, or 'Noise' if not found
df_chunks_for_plot['topic_name'] = df_chunks_for_plot['topic_name'].replace(original_noise_topic_generated_name, 'Noise (Topic -1)')
# Also replace any literal 'Noise' if it somehow ended up there (e.g. from fillna for other missing topic names)
df_chunks_for_plot['topic_name'] = df_chunks_for_plot['topic_name'].replace('Noise', 'Noise (Topic -1)')

# Re-ensure topic is categorical for distinct colors in plotly, in case replacement changed its type
df_chunks_for_plot['topic_name'] = df_chunks_for_plot['topic_name'].astype('category')

# Group by year and topic_name and count occurrences
topic_evolution = df_chunks_for_plot.groupby(['year', 'topic_name']).size().reset_index(name='count')

# The 'Noise' topic (ID -1) is mapped to its generated name (e.g., '-1_great_right_president_new') or 'Noise'
# We will not exclude it, so it is included in the topic evolution.
# Removed: topic_evolution = topic_evolution[topic_evolution['topic_name'] != '-1_great_right_america_new']

# Calculate the total number of chunks per year
yearly_total_chunks = topic_evolution.groupby('year')['count'].sum().reset_index(name='total_chunks')

# Merge to get the total chunks per year for percentage calculation
topic_evolution = pd.merge(topic_evolution, yearly_total_chunks, on='year')

# Calculate the percentage of each topic per year
topic_evolution['percentage'] = (topic_evolution['count'] / topic_evolution['total_chunks']) * 100

# Calculate the total percentage for each topic to order them by magnitude across all years
topic_order = topic_evolution.groupby('topic_name')['percentage'].sum().sort_values(ascending=False).index.tolist()

# Create a stacked bar chart to visualize topic evolution per year (in percentages)
fig = px.bar(
    topic_evolution,
    x='year',
    y='percentage',
    color='topic_name',
    title='Topic Evolution Over Years (Percentage of Chunks per Topic per Year)',
    labels={
        'year': 'Year',
        'percentage': 'Percentage of Chunks',
        'topic_name': 'Topic'
    },
    category_orders={
        'year': sorted(topic_evolution['year'].unique()),
        'topic_name': topic_order  # Order topics by magnitude
    }
)

fig.update_layout(
    xaxis_tickvals=topic_evolution['year'].unique(),
    xaxis_ticktext=[str(y) for y in sorted(topic_evolution['year'].unique())],
    xaxis_type='category',
    bargap=0.3,
    height=600,
    width=900
)

fig.write_html(get_figure_path("bertopic_auto_yearsevolution.html"))
fig.show()

In [ ]:
import plotly.express as px

# Define month order for consistent plotting
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# Standardize the name for the Noise topic (Topic ID -1)
# The topic_name_mapping from previous cells is used to get the original BERTopic generated name for topic -1.
# This ensures that all instances of topic ID -1, whether initially named by BERTopic or as 'Noise' by fillna,
# are consolidated under a single, user-friendly label.
original_noise_topic_generated_name = topic_name_mapping.get(-1, 'Noise') # Get the generated name, or 'Noise' if not found
df_chunks_for_plot['topic_name'] = df_chunks_for_plot['topic_name'].replace(original_noise_topic_generated_name, 'Noise (Topic -1)')
# Also replace any literal 'Noise' if it somehow ended up there (e.g. from fillna for other missing topic names)
df_chunks_for_plot['topic_name'] = df_chunks_for_plot['topic_name'].replace('Noise', 'Noise (Topic -1)')

# Re-ensure topic is categorical for distinct colors in plotly, in case replacement changed its type
df_chunks_for_plot['topic_name'] = df_chunks_for_plot['topic_name'].astype('category')

# Group by month and topic_name and count occurrences
topic_evolution_monthly = df_chunks_for_plot.groupby(['month', 'topic_name']).size().reset_index(name='count')

# Removed: Exclude the 'Noise' topic (-1) from this analysis as it is now unified and included
# topic_evolution_monthly = topic_evolution_monthly[topic_evolution_monthly['topic_name'] != '-1_great_right_america_new']

# Calculate the total number of chunks per month
monthly_total_chunks = topic_evolution_monthly.groupby('month')['count'].sum().reset_index(name='total_chunks')

# Merge to get the total chunks per month for percentage calculation
topic_evolution_monthly = pd.merge(topic_evolution_monthly, monthly_total_chunks, on='month')

# Calculate the percentage of each topic per month
topic_evolution_monthly['percentage'] = (topic_evolution_monthly['count'] / topic_evolution_monthly['total_chunks']) * 100

# Ensure the months are ordered correctly for plotting
topic_evolution_monthly['month'] = pd.Categorical(topic_evolution_monthly['month'], categories=month_order, ordered=True)
topic_evolution_monthly = topic_evolution_monthly.sort_values('month')

# Create a stacked bar chart to visualize topic evolution per month (in percentages)
fig = px.bar(
    topic_evolution_monthly,
    x='month',
    y='percentage',
    color='topic_name',
    title='Topic Evolution Over Months (Percentage of Chunks per Topic per Month)',
    labels={
        'month': 'Month',
        'percentage': 'Percentage of Chunks',
        'topic_name': 'Topic'
    },
    category_orders={'month': month_order} # Use predefined month order
)

fig.update_layout(
    xaxis_type='category',
    bargap=0.3,
    height=600,
    width=900
)

fig.write_html(get_figure_path("bertopic_auto_monthevolution.html"))
fig.show()

In [ ]:
import plotly.express as px

# Group by location and topic_name and count occurrences
topic_evolution_location = df_chunks_for_plot.groupby(['location', 'topic_name']).size().reset_index(name='count')

# The 'Noise' topic (ID -1) is mapped to its generated name (e.g., '-1_great_right_president_new') or 'Noise'
# We will not exclude it, so it is included in the topic evolution.
# Removed: topic_evolution_location = topic_evolution_location[topic_evolution_location['topic_name'] != '-1_great_right_america_new']

# Calculate the total number of chunks per location
location_total_chunks = topic_evolution_location.groupby('location')['count'].sum().reset_index(name='total_chunks')

# Merge to get the total chunks per location for percentage calculation
topic_evolution_location = pd.merge(topic_evolution_location, location_total_chunks, on='location')

# Calculate the percentage of each topic per location
topic_evolution_location['percentage'] = (topic_evolution_location['count'] / topic_evolution_location['total_chunks']) * 100

# Sort locations by the total number of chunks, or alphabetically for consistency
topic_evolution_location = topic_evolution_location.sort_values(by=['location', 'percentage'], ascending=[True, False])

# Create a stacked bar chart to visualize topic evolution per location (in percentages)
fig = px.bar(
    topic_evolution_location,
    x='location',
    y='percentage',
    color='topic_name',
    title='Topic Evolution Over Locations (Percentage of Chunks per Topic per Location)',
    labels={
        'location': 'Location',
        'percentage': 'Percentage of Chunks',
        'topic_name': 'Topic'
    },
    category_orders={'location': sorted(topic_evolution_location['location'].unique())}
)

fig.update_layout(
    xaxis_type='category',
    bargap=0.3,
    height=600,
    width=1000
)

fig.write_html(get_figure_path("bertopic_auto_locationevolution.html"))
fig.show()

In [ ]:
import plotly.express as px

# Filter data for 2019 and 2020
topic_evolution_filtered = topic_evolution[topic_evolution['year'].isin([2019, 2020])]

# Create a grouped bar chart for topic distribution per year
fig = px.bar(
    topic_evolution_filtered,
    x='topic_name',
    y='percentage',
    color='year',
    barmode='group',
    title='Topic Distribution by Year (2019 vs 2020)',
    labels={
        'topic_name': 'Topic',
        'percentage': 'Percentage of Chunks',
        'year': 'Year'
    },
    category_orders={'year': [2019, 2020]} # Ensure years are ordered
)

fig.update_layout(
    xaxis_tickangle=-45,
    height=700,
    width=1000,
    legend_title_text='Year'
)

fig.write_html(get_figure_path("bertopic_auto_years.html"))
fig.show()

In [ ]:
print("Saved output files:\n")

for root, dirs, files in os.walk(BASE_OUTPUT_DIR):
    level = root.replace(BASE_OUTPUT_DIR, "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 4 * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

In [ ]:
import shutil

# creates a zip with the original folder and tthe new ones
project_folder = BASE_PROJECT_DIR
zip_output_name = "TAM_2526_Project_final"

shutil.make_archive(
    zip_output_name,
    "zip",
    project_folder
)

print(f"Final zip created: {zip_output_name}.zip")

In [ ]:
from google.colab import files

files.download("TAM_2526_Project_final.zip")